In [1]:
!pip install -q -U torch transformers datasets peft bitsandbytes trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5

In [2]:
import torch
import os
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 0. Disable W&B logging
os.environ["WANDB_DISABLED"] = "true"

# 1. Load and Format SQuAD Dataset
print("Loading dataset...")
dataset = load_dataset("squad", split="train[:2000]") # Small slice for demonstration

def format_squad_for_phi3(sample):
    # Standard Phi-3 Instruct format
    return {
        "text": f"<|user|>\nContext: {sample['context']}\n\nQuestion: {sample['question']}<|end|>\n<|assistant|>\n{sample['answers']['text'][0]}<|end|>"
    }

dataset = dataset.map(format_squad_for_phi3)

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
# 2. Load Model & Tokenizer
model_id = "microsoft/Phi-3.5-mini-instruct"
print(f"Loading {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Compute in float16 (fast on T4)
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
    torch_dtype=torch.float16 # Request Float16 weights
)

Loading microsoft/Phi-3.5-mini-instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [4]:
# 3. Prepare for Training
# This casts LayerNorms to float32 for stability
model = prepare_model_for_kbit_training(model)

# 4. Attach LoRA Adapters
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)

# 5. Configure Training (T4 STABILITY MODE)
# We disable fp16 to avoid the "Scaler" crash.
training_args = SFTConfig(
    output_dir="./phi3-squad-finetune",
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=60,
    fp16=False,
    bf16=False,
    optim="paged_adamw_32bit",
    save_strategy="no",
    report_to="none"
)

# 6. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer
)

# 7. Start Training
print("Starting training (Float32 Optimizer Mode)...")
model.config.use_cache = False
trainer.train()
print("Training finished!")

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Starting training (Float32 Optimizer Mode)...


Step,Training Loss
5,1.932900
10,1.685600
15,1.623900
20,1.612100
25,1.661500
30,1.633700
35,1.596500
40,1.467600
45,1.450100
50,1.642400


Training finished!


In [5]:
# 1. Switch model to evaluation mode
model.eval()

# 2. Define a test sample
test_context = "The Apollo program was a human spaceflight program carried out by NASA. It succeeded in landing the first humans on the Moon from 1969 to 1972."
test_question = "When did the Apollo program land humans on the Moon?"

# 3. Format prompt
prompt = f"<|user|>\nContext: {test_context}\n\nQuestion: {test_question}<|end|>\n<|assistant|>"

# 4. Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 5. Generate Response
print(f"Question: {test_question}")
print("Generating response...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        temperature=0.0
    )

# 6. Decode
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer_part = response.split("<|assistant|>")[-1].strip()

print("\n" + "="*30)
print(f"Model Answer:\n{answer_part}")
print("="*30)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: When did the Apollo program land humans on the Moon?
Generating response...

Model Answer:
Context: The Apollo program was a human spaceflight program carried out by NASA. It succeeded in landing the first humans on the Moon from 1969 to 1972.

Question: When did the Apollo program land humans on the Moon? 1969 to 1972


In [7]:
from google.colab import drive
import shutil
import os

# --- Step A: Save to a local folder in Colab first ---
local_save_path = "./phi3_squad_final_local"
print(f"1. Saving model adapter files locally to: {local_save_path}")
trainer.save_model(local_save_path)
tokenizer.save_pretrained(local_save_path)
print("   Local save complete.")

# --- Step B: Copy that folder to Google Drive ---
drive.mount('/content/drive')
drive_save_path = "/content/drive/My Drive/ML_Project_Phase2/phi3_squad_final"

print(f"2. Copying files to Google Drive: {drive_save_path}")

if os.path.exists(drive_save_path):
    print("   Existing folder found in Drive. Removing it to ensure fresh copy...")
    shutil.rmtree(drive_save_path)

shutil.copytree(local_save_path, drive_save_path)

print("\nSUCCESS: Full model saved permanently to Google Drive!")

1. Saving model adapter files locally to: ./phi3_squad_final_local
   Local save complete.
Mounted at /content/drive
2. Copying files to Google Drive: /content/drive/My Drive/ML_Project_Phase2/phi3_squad_final

SUCCESS: Full model saved permanently to Google Drive!
